# 06 – Projections: where in "input space" does the model fail, and why there?

We describe every test input as a vector, project the vectors to 2D, and look for regions where errors pile up – then
ask what distinguishes those regions, and whether they split into sub-regions where the model behaves differently.

Two ways to describe an input:

* **A – surface form:** which short character sequences (2–4 characters) the input string contains. Inputs that *look*
  alike end up close together. Variable names are normalized first, keeping their *role* (the data always writes `x`
  as the left and `y` as the right operand; a third letter takes the free role) – otherwise the layout simply sorts
  inputs by their variable letters. The original variable pair is still available for colouring.
* **B – building blocks:** which bitwise terms the input is made of (`x&y`, `~x|y`, `x^y`, …) and with what
  coefficients, ignoring their order. Inputs made of the same ingredients end up close together.

**Structure – overview first, zoom, details on demand:**
1 the two descriptions · 2 where the errors are · 3 **overview: a few large regions, each named by its content** ·
4 zoom: fine clusters and sub-clusters · 5 **wrong outputs only: kinds of failure** ·
6 **correct is not the same as robust** (uses `05`) · 7 same ingredients, different outcome ·
8 interactive explorer (`figures/projection_explorer.html`).

Earlier results are reused as colours: mistake types from `04` (`taxonomy.csv`) and the input changes from `05`
(`probe_results_full.csv` if you ran `05` with `FULL = True`, otherwise the 1,500-input sample `probe_results.csv`).

Needs: `pip install umap-learn scikit-learn` (UMAP takes ~1 min per projection on a laptop).

In [ ]:
import ast
import itertools
import json
import re
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import HDBSCAN
import umap

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "predictions_test.csv").exists())
sys.path.insert(0, str(ROOT / "src"))
import neureduce as nr

FIG = Path.cwd() / "figures"
FIG.mkdir(exist_ok=True)
SEED = 0

df = pd.read_csv(ROOT / "data" / "predictions_test.csv").reset_index(drop=True)
train = pd.read_csv(ROOT / "data" / "predictions_train.csv")
df["error"] = ~df["correct"]
df["trg_train_freq"] = df.trg.map(Counter(train.trg)).fillna(0).astype(int)
df["var_pair"] = ["".join(sorted(set(re.findall(r"[a-z]", s)))) for s in df.src]
df["answer_kind"] = np.select(
    [df.src_n_vars == 3, (df.trg_n_terms == 1) & (df.trg_len <= 5) & (df.trg_n_vars == 2), df.trg_len >= 15],
    ["3-variable input", "two-variable bitwise answer", "long answer (>= 15)"], "other")
BASE = df.error.mean()
print(f"{len(df):,} test inputs, error rate {BASE:.1%}")

### Earlier results attached to every input

* `mistake`: the mistake type from `04` for wrong outputs (`correct` otherwise).
* `robustness`: for inputs the model solved, the share of the eight meaning-preserving changes from `05` after which
  it is **still** correct (1 = survives everything, 0 = breaks under every change).
* `probe: <change>`: per change, whether it broke a correct answer (or fixed a wrong one).

In [ ]:
here = Path.cwd()
df["mistake"] = np.where(df.correct, "correct", np.where(df.verdict == "invalid", "unparseable output", "(run 04)"))
if (here / "taxonomy.csv").exists():
    tax = pd.read_csv(here / "taxonomy.csv").set_index("row").category
    has = df.row.isin(tax.index)
    df.loc[has, "mistake"] = df.loc[has, "row"].map(tax)
else:
    print("taxonomy.csv not found - run 04_error_taxonomy.ipynb to the end first")

probe_file = here / "probe_results_full.csv"
if not probe_file.exists():
    probe_file = here / "probe_results.csv"
    print("using the 1,500-input probe sample - run 05 with FULL = True to cover every input")
pr = pd.read_csv(probe_file)
pr = pr[pr.probe != "control (no change)"].copy()
pr["ok"] = pr.new_verdict.isin(["exact", "equivalent"])
PROBE_NAMES = list(dict.fromkeys(pr.probe))
df["robustness"] = df.row.map(pr[pr.orig == "correct"].groupby("row").ok.mean())
for p in PROBE_NAMES:
    res = pr[pr.probe == p].set_index("row")
    labels = np.where(res.orig == "correct", np.where(res.ok, "still correct", "broken"),
                      np.where(res.ok, "now correct", "still wrong"))
    df[f"probe: {p}"] = df.row.map(pd.Series(labels, index=res.index)).fillna("not tested / n.a.")
print(f"{probe_file.name}: robustness known for {df.robustness.notna().sum():,} correctly solved inputs")
print(df.mistake.value_counts().to_dict())

## 1. Two descriptions of each input

In [ ]:
# A - surface form: character 2-4-grams, TF-IDF weighted
# dataset convention: inside a bitwise operation, which variable is written on the left / right?
pos = Counter()
for s in train.src:
    for a, b in re.findall(r"~?\(?~?([a-z])[&|^]\(?~?([a-z])", s):
        if a != b:
            pos[("x", "left")] += a == "x"; pos[("x", "right")] += b == "x"
            pos[("y", "left")] += a == "y"; pos[("y", "right")] += b == "y"
print("training inputs - operand positions:", dict(pos))


def normalize_names(expr):
    """Keep x and y; map any other letter to the role (x = left, y = right) that is free in this input."""
    names = set(re.findall(r"[a-z]", expr))
    others = sorted(names - {"x", "y", "z"})
    free = [v for v in "xy" if v not in names]
    return expr.translate(str.maketrans(dict(zip(others, free))))

df["src_normalized"] = df.src.map(normalize_names)
tfidf = TfidfVectorizer(analyzer="char", ngram_range=(2, 4), min_df=5, sublinear_tf=True)
X_surface = tfidf.fit_transform(df.src_normalized)


# B - building blocks: coefficient of every bitwise term (identified by its variables + truth table), plus the constant
BLOCK_LABEL = {}

def building_blocks(expr):
    feats = Counter()

    def walk(n, sign):
        if isinstance(n, ast.BinOp) and isinstance(n.op, (ast.Add, ast.Sub)):
            walk(n.left, sign)
            walk(n.right, sign if isinstance(n.op, ast.Add) else -sign)
        elif nr._const(n):
            feats["constant"] += sign * nr.evaluate(n, {})
        elif isinstance(n, ast.UnaryOp) and isinstance(n.op, (ast.USub, ast.UAdd)):
            walk(n.operand, -sign if isinstance(n.op, ast.USub) else sign)
        elif nr._bitwise(n):
            text = ast.unparse(n).replace(" ", "")
            names = sorted(set(re.findall(r"[a-z]", text)))
            tt = "".join(str(nr.evaluate(n, dict(zip(names, b))) & 1)
                         for b in itertools.product((0, 1), repeat=len(names)))
            key = f"{''.join(names)}:{tt}"
            BLOCK_LABEL.setdefault(key, text)
            feats[key] += sign
        elif isinstance(n, ast.BinOp) and isinstance(n.op, ast.Mult) and nr._const(n.left):
            walk(n.right, sign * nr.evaluate(n.left, {}))
        elif isinstance(n, ast.BinOp) and isinstance(n.op, ast.Mult) and nr._const(n.right):
            walk(n.left, sign * nr.evaluate(n.right, {}))
        else:
            raise ValueError(expr)

    walk(nr.parse(expr), 1)
    return {k: v for k, v in feats.items() if v}

blocks = [building_blocks(s) for s in df.src]
dv = DictVectorizer()
X_blocks = dv.fit_transform(blocks)
print(f"A: {X_surface.shape[1]:,} character n-grams | B: {X_blocks.shape[1]:,} distinct building blocks")
print("example:", df.src[0], "->", {BLOCK_LABEL.get(k, k): v for k, v in blocks[0].items()})

In [ ]:
def project(X, n_components=40):
    Z = TruncatedSVD(n_components=min(n_components, X.shape[1] - 1), random_state=SEED).fit_transform(X)
    return umap.UMAP(n_neighbors=30, min_dist=0.1, random_state=SEED, n_jobs=1).fit_transform(Z)

df[["sx", "sy"]] = project(X_surface)
df[["bx", "by"]] = project(X_blocks)
print("done")

## 2. Where are the errors?

Left: every input, wrong ones in red on top. Right: the same layout as an error-rate map (hexagons with at least 10
inputs; darker = more errors). The right view is the one to read – red dots alone overplot.

In [ ]:
def error_maps(xcol, ycol, title, fname):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
    ok, bad = df[~df.error], df[df.error]
    axes[0].scatter(ok[xcol], ok[ycol], s=2, color="lightgray", label=f"correct ({len(ok):,})")
    axes[0].scatter(bad[xcol], bad[ycol], s=4, color="tab:red", label=f"wrong ({len(bad):,})")
    axes[0].legend(markerscale=4, fontsize=8)
    hb = axes[1].hexbin(df[xcol], df[ycol], C=df.error.astype(float), reduce_C_function=np.mean,
                        gridsize=45, mincnt=10, cmap="Reds", vmin=0, vmax=0.5)
    fig.colorbar(hb, ax=axes[1], label="error rate (capped at 50%)")
    for ax in axes:
        ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(FIG / fname, dpi=200)

error_maps("sx", "sy", "A – surface form (inputs that look alike are close)", "proj_surface_errors.png")
error_maps("bx", "by", "B – building blocks (inputs made of the same terms are close)", "proj_blocks_errors.png")

## 3. Overview: a few large regions, named by their content

Each map is split into `N_REGIONS` (default 8) contiguous regions (k-means on the 2D layout) – few enough to explain each one. Each region gets an automatic
name from what makes it different: its variables, or the building blocks that are much more common inside than outside.
The table adds what we know from `03`–`05` about each region.

In [ ]:
PRESENT = (X_blocks != 0).astype(float).tocsc()
BLOCK_NAMES = dv.get_feature_names_out()


def auto_label(mask, k=2):
    """Human-readable name for a group of inputs: variables, over-represented building blocks, unusual size."""
    d = df[mask]
    if (d.src_n_vars == 3).mean() > 0.8:
        return "3 variables"
    parts = ["other variable pair"] if (d.var_pair != "xy").mean() > 0.8 else []
    if not parts:
        inside = np.asarray(PRESENT[mask.to_numpy()].mean(axis=0)).ravel()
        outside = np.asarray(PRESENT[~mask.to_numpy()].mean(axis=0)).ravel()
        parts += ["has " + BLOCK_LABEL.get(BLOCK_NAMES[j], BLOCK_NAMES[j])
                  for j in np.argsort(inside - outside)[::-1][:k]
                  if inside[j] >= 0.5 and inside[j] - outside[j] >= 0.25 and BLOCK_NAMES[j] != "constant"]
    for colname, low, high in [("src_n_terms", "few input terms", "many input terms"),
                               ("trg_len", "short answers", "long answers")]:
        z = (d[colname].mean() - df[colname].mean()) / df[colname].std()
        if abs(z) > 0.5:
            parts.append(high if z > 0 else low)
    return ", ".join(parts[:3]) if parts else "typical inputs"


def region_table(col):
    rows = []
    for cid in sorted(c for c in df[col].unique() if c >= 0):
        m = df[col] == cid
        d = df[m]
        mist = d.loc[d.error, "mistake"].value_counts()
        rows.append({"region": cid, "name": auto_label(m), "n": len(d), "error_rate": d.error.mean(),
                     "robustness": d.robustness.mean(), "answer_unseen": (d.trg_train_freq == 0).mean(),
                     "top_mistake": f"{mist.index[0]} ({mist.iloc[0] / mist.sum():.0%})" if len(mist) else "-"})
    return pd.DataFrame(rows).set_index("region").sort_values("error_rate", ascending=False)


def overview_map(xcol, ycol, col, table, title, fname):
    fig, ax = plt.subplots(figsize=(9, 7))
    noise = df[col] < 0
    ax.scatter(df.loc[noise, xcol], df.loc[noise, ycol], s=2, color="#e8e8e8")
    sc = ax.scatter(df.loc[~noise, xcol], df.loc[~noise, ycol], s=3, cmap="Reds", vmin=0, vmax=0.5,
                    c=df.loc[~noise, col].map(table.error_rate))
    for cid, r in table.iterrows():
        m = df[col] == cid
        ax.annotate(f"{r['name']}\n{r.error_rate:.0%} wrong · n={int(r.n):,}",
                    (df.loc[m, xcol].median(), df.loc[m, ycol].median()), fontsize=7.5, ha="center",
                    bbox=dict(boxstyle="round", fc="white", alpha=0.85))
    fig.colorbar(sc, ax=ax, label="region error rate (capped at 50%)")
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=10)
    fig.tight_layout()
    fig.savefig(FIG / fname, dpi=200)


from sklearn.cluster import KMeans

N_REGIONS = 8  # few, large, contiguous regions - change to zoom out/in
for xc, yc, col in [("sx", "sy", "r_surface"), ("bx", "by", "r_blocks")]:
    df[col] = KMeans(n_clusters=N_REGIONS, n_init=10, random_state=SEED).fit_predict(df[[xc, yc]].to_numpy())
reg_s, reg_b = region_table("r_surface"), region_table("r_blocks")
overview_map("sx", "sy", "r_surface", reg_s, "A – surface form: overview regions", "proj_surface_overview.png")
reg_s.round(3)

In [ ]:
overview_map("bx", "by", "r_blocks", reg_b, "B – building blocks: overview regions", "proj_blocks_overview.png")
reg_b.round(3)

## 4. Zoom in: fine clusters and what distinguishes them

Finer clustering of each layout (HDBSCAN, label -1 = "noise", not part of any cluster). For each cluster: its size,
error rate, and how it differs from the rest – variable pair, kind of answer, how often its answers occurred in training.

In [ ]:
def find_clusters(xcol, ycol, min_size=80):
    return HDBSCAN(min_cluster_size=min_size, min_samples=10, copy=True).fit_predict(df[[xcol, ycol]].to_numpy())

df["c_surface"] = find_clusters("sx", "sy")
df["c_blocks"] = find_clusters("bx", "by")


def describe(d):
    top_pair, top_trg = d.var_pair.value_counts(), d.trg.value_counts()
    return pd.Series({
        "n": len(d), "error_rate": d.error.mean(), "errors": int(d.error.sum()),
        "top_vars": f"{top_pair.index[0]} ({top_pair.iloc[0] / len(d):.0%})",
        "top_answer": f"{top_trg.index[0]} ({top_trg.iloc[0] / len(d):.0%})",
        "3var": (d.src_n_vars == 3).mean(), "bitwise_answer": (d.answer_kind == "two-variable bitwise answer").mean(),
        "src_len": d.src_len.mean(), "src_terms": d.src_n_terms.mean(), "answer_unseen": (d.trg_train_freq == 0).mean(),
    })


def cluster_table(col):
    t = df[df[col] >= 0].groupby(col).apply(describe, include_groups=False)
    return t.sort_values("error_rate", ascending=False)

pd.set_option("display.width", 200)
print(f"overall error rate {BASE:.1%}")
tab_s = cluster_table("c_surface")
tab_s.head(10).round(3)

In [ ]:
tab_b = cluster_table("c_blocks")
tab_b.head(10).round(3)

In [ ]:
# a side question the cluster tables raise: inputs with the usual pair x,y vs. any other pair
d = df.assign(pair=np.where(df.var_pair == "xy", "x,y", np.where(df.src_n_vars == 3, "3 variables", "other pair")))
d.groupby("pair").agg(n=("error", "size"), error_rate=("error", "mean"),
                      answer_unseen=("trg_train_freq", lambda f: (f == 0).mean())).round(3)

In [ ]:
def cluster_map(xcol, ycol, ccol, table, title, fname, top=6):
    fig, ax = plt.subplots(figsize=(8, 6.5))
    noise = df[ccol] < 0
    ax.scatter(df.loc[noise, xcol], df.loc[noise, ycol], s=2, color="#e5e5e5")
    sc = ax.scatter(df.loc[~noise, xcol], df.loc[~noise, ycol], s=3, c=df.loc[~noise, ccol].map(table.error_rate),
                    cmap="Reds", vmin=0, vmax=0.5)
    for cid in table.index[:top]:  # label the worst clusters
        m = df[ccol] == cid
        ax.annotate(f"#{cid}: {table.error_rate[cid]:.0%}\n(n={int(table.n[cid])})",
                    (df.loc[m, xcol].median(), df.loc[m, ycol].median()), fontsize=8, ha="center",
                    bbox=dict(boxstyle="round", fc="white", alpha=0.8))
    fig.colorbar(sc, ax=ax, label="cluster error rate (capped at 50%)")
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=10)
    fig.tight_layout()
    fig.savefig(FIG / fname, dpi=200)

cluster_map("sx", "sy", "c_surface", tab_s, "A – clusters coloured by error rate (worst labelled)", "proj_surface_clusters.png")
cluster_map("bx", "by", "c_blocks", tab_b, "B – clusters coloured by error rate (worst labelled)", "proj_blocks_clusters.png")

### What sets the worst cluster apart?

For the worst cluster of each layout: the character sequences (A) or building blocks (B) that are much more common
inside the cluster than outside. This is the "what distinguishes this cluster?" question from the assignment.

In [ ]:
def distinguishing_ngrams(mask, k=12):
    inside = np.asarray(X_surface[mask.to_numpy()].mean(axis=0)).ravel()
    outside = np.asarray(X_surface[~mask.to_numpy()].mean(axis=0)).ravel()
    names = tfidf.get_feature_names_out()
    order = np.argsort(inside - outside)[::-1][:k]
    return pd.DataFrame({"ngram": names[order], "inside": inside[order], "outside": outside[order]}).round(3)


def distinguishing_blocks(mask, k=12):
    present = (X_blocks != 0).astype(float)
    inside = np.asarray(present[mask.to_numpy()].mean(axis=0)).ravel()
    outside = np.asarray(present[~mask.to_numpy()].mean(axis=0)).ravel()
    names = dv.get_feature_names_out()
    order = np.argsort(inside - outside)[::-1][:k]
    return pd.DataFrame({"block": [BLOCK_LABEL.get(n, n) for n in names[order]],
                         "share_inside": inside[order], "share_outside": outside[order]}).round(3)

worst_s = tab_s[tab_s.n >= 100].index[0]
print(f"A: worst cluster #{worst_s} (error {tab_s.error_rate[worst_s]:.0%}, n={int(tab_s.n[worst_s])})")
display(distinguishing_ngrams(df.c_surface == worst_s))
worst_b = tab_b[tab_b.n >= 100].index[0]
print(f"B: worst cluster #{worst_b} (error {tab_b.error_rate[worst_b]:.0%}, n={int(tab_b.n[worst_b])})")
distinguishing_blocks(df.c_blocks == worst_b)

### Sub-clusters: does the model behave differently inside the worst cluster?

Re-cluster only the points of the worst building-block cluster, then compare the sub-clusters.

In [ ]:
def subclusters(ccol, cid, xcol, ycol, min_size=None):
    m = df[ccol] == cid
    sub = df[m].copy()
    min_size = min_size or max(10, len(sub) // 8)  # scale with the cluster size
    sub["sub"] = HDBSCAN(min_cluster_size=min_size, min_samples=5, copy=True).fit_predict(sub[[xcol, ycol]].to_numpy())
    found = sub[sub["sub"] >= 0]
    if found["sub"].nunique() < 2:
        print(f"No sub-structure: HDBSCAN finds {found['sub'].nunique()} sub-cluster(s) inside #{cid} "
              f"(n={len(sub)}) - the cluster is homogeneous at this resolution. That is an answer too.")
        return sub, pd.DataFrame()
    t = found.groupby("sub").apply(describe, include_groups=False).sort_values("error_rate", ascending=False)
    fig, ax = plt.subplots(figsize=(6.5, 5))
    for s_id in sorted(sub["sub"].unique()):
        p = sub[sub["sub"] == s_id]
        label = "noise" if s_id < 0 else f"sub {s_id}: {t.error_rate[s_id]:.0%} (n={len(p)})"
        ax.scatter(p[xcol], p[ycol], s=8, label=label, color="#dddddd" if s_id < 0 else None)
    ax.legend(fontsize=7, markerscale=2)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"Inside cluster #{cid}", fontsize=10)
    fig.tight_layout()
    fig.savefig(FIG / f"proj_subclusters_{ccol}_{cid}.png", dpi=200)
    return sub, t

sub_b, sub_tab = subclusters("c_blocks", worst_b, "bx", "by")
sub_tab.round(3)

In [ ]:
# read examples from the best and the worst sub-cluster
pd.set_option("display.max_colwidth", 60)
if len(sub_tab) >= 2:
    for s_id in [sub_tab.index[0], sub_tab.index[-1]]:
        print(f"sub-cluster {s_id}: error {sub_tab.error_rate[s_id]:.0%}")
        display(sub_b[sub_b["sub"] == s_id][["src", "trg", "pred", "verdict"]].head(6))

## 5. Kinds of failure: a projection of the wrong outputs only

Here each **wrong output** is described by *how it differs from the correct answer*: the building blocks the model wrote
too much (+) or too little (−) compared to the target. `x&y` → `x` becomes "+x −x&y". Wrong outputs with the same kind
of difference end up close together, so clusters are recurring mistakes – each named by its most common difference.

In [ ]:
def fmt_diff(diff):
    parts = []
    for key, v in sorted(diff.items(), key=lambda kv: (-abs(kv[1]), kv[0])):
        name = "const" if key == "constant" else BLOCK_LABEL.get(key, key)
        parts.append(f"{'+' if v > 0 else '−'}{'' if abs(v) == 1 else f'{abs(v)}·'}{name}")
    return " ".join(parts)


err = df[df.verdict == "wrong"].copy()
diffs, keep = [], []
for i, p, t in zip(err.index, err.pred, err.trg):
    try:
        d = Counter(building_blocks(p))
        d.subtract(building_blocks(t))
        diffs.append({k: v for k, v in d.items() if v})
        keep.append(i)
    except ValueError:  # non-linear output
        pass
err = err.loc[keep]
err["diff"] = [fmt_diff(d) for d in diffs]
X_diff = DictVectorizer().fit_transform(diffs)
Z = TruncatedSVD(n_components=min(20, X_diff.shape[1] - 1), random_state=SEED).fit_transform(X_diff)
err[["ex", "ey"]] = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=SEED, n_jobs=1).fit_transform(Z)
err["e_cluster"] = HDBSCAN(min_cluster_size=15, min_samples=5, copy=True).fit_predict(err[["ex", "ey"]].to_numpy())

fail = (err[err.e_cluster >= 0].groupby("e_cluster")
        .apply(lambda d: pd.Series({
            "n": len(d),
            "most_common_difference": f"{d['diff'].value_counts().index[0]}  ({d['diff'].value_counts().iloc[0]}/{len(d)})",
            "mistake_type": d.mistake.value_counts().index[0],
            "example (answer -> model)": f"{d.trg.iloc[0]}  ->  {d.pred.iloc[0]}"}), include_groups=False)
        .sort_values("n", ascending=False))
err["e_region"] = [fail.most_common_difference[c] if c >= 0 else "unclustered" for c in err.e_cluster]

fig, ax = plt.subplots(figsize=(10, 7))
for k, cat in enumerate(sorted(err.mistake.unique())):
    p = err[err.mistake == cat]
    ax.scatter(p.ex, p.ey, s=10, alpha=0.75, color=plt.get_cmap("tab10").colors[k % 10], label=f"{cat} ({len(p)})")
for cid in fail.index[:10]:
    m = err.e_cluster == cid
    ax.annotate(fail.most_common_difference[cid], (err.loc[m, "ex"].median(), err.loc[m, "ey"].median()),
                fontsize=7, ha="center", bbox=dict(boxstyle="round", fc="white", alpha=0.85))
ax.legend(fontsize=7, markerscale=1.5)
ax.set_xticks([]); ax.set_yticks([])
ax.set_title(f"{len(err)} wrong outputs, placed by how they differ from the correct answer\n"
             "(colour = mistake type from 04; labels = most common difference in the 10 largest clusters)", fontsize=9)
fig.tight_layout()
fig.savefig(FIG / "proj_failures.png", dpi=200)
pd.set_option("display.max_colwidth", 50)
fail.head(15)

## 6. Correct is not the same as robust

Colour = robustness of each correctly solved input (share of the eight changes from `05` it survives). If fragile inputs
sit in regions where the model is usually *right*, its accuracy there depends on inputs being written the usual way –
something no accuracy number shows.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
known = df.robustness.notna()
for ax, (xc, yc, name) in zip(axes, [("sx", "sy", "A – surface form"), ("bx", "by", "B – building blocks")]):
    ax.scatter(df.loc[~known, xc], df.loc[~known, yc], s=2, color="#e8e8e8")
    sc = ax.scatter(df.loc[known, xc], df.loc[known, yc], s=4, c=df.loc[known, "robustness"], cmap="RdYlGn",
                    vmin=0, vmax=1)
    ax.set_title(name, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(sc, ax=axes, label="share of meaning-preserving changes survived")
fig.suptitle("Robustness of correctly solved inputs (grey = wrong or not tested)")
fig.savefig(FIG / "proj_robustness.png", dpi=200, bbox_inches="tight")

In [ ]:
# per overview region: error rate vs. robustness - are the fragile regions the error regions?
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for ax, (tab, name) in zip(axes, [(reg_s, "A – surface regions"), (reg_b, "B – building-block regions")]):
    ax.scatter(tab.error_rate, tab.robustness, s=tab.n / 8, alpha=0.6, color="tab:purple")
    for _, r in tab.iterrows():
        ax.annotate(r["name"], (r.error_rate, r.robustness), fontsize=7, xytext=(4, 4), textcoords="offset points")
    ax.set_xlabel("error rate of the region")
    ax.set_ylabel("mean robustness of its correct answers")
    ax.set_title(f"{name} (size = n; r = {tab.error_rate.corr(tab.robustness):.2f})", fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(FIG / "proj_regions_error_vs_robustness.png", dpi=200)

In [ ]:
# which change breaks which region? share of correct answers that survive, rows = building-block regions
def survival(s):
    tested = s.isin(["still correct", "broken"])
    return (s == "still correct").sum() / tested.sum() if tested.any() else np.nan

surv = pd.DataFrame({p: df[df.r_blocks >= 0].groupby("r_blocks")[f"probe: {p}"].apply(survival) for p in PROBE_NAMES})
surv = surv.loc[reg_b.index]
surv.index = [f"{reg_b.name[i]} ({reg_b.error_rate[i]:.0%} wrong)" for i in surv.index]
fig, ax = plt.subplots(figsize=(1.1 * len(PROBE_NAMES) + 3.5, 0.45 * len(surv) + 2))
im = ax.imshow(surv.to_numpy(dtype=float), cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
for a in range(surv.shape[0]):
    for b in range(surv.shape[1]):
        ax.text(b, a, f"{surv.iat[a, b]:.0%}", ha="center", va="center", fontsize=7)
ax.set_xticks(range(len(PROBE_NAMES)), PROBE_NAMES, rotation=35, ha="right", fontsize=8)
ax.set_yticks(range(len(surv)), surv.index, fontsize=8)
ax.set_title("Share of correct answers that survive each change, per region (B)", fontsize=10)
fig.colorbar(im, ax=ax, fraction=0.03)
fig.tight_layout()
fig.savefig(FIG / "proj_regions_by_change.png", dpi=200)

## 7. Same ingredients, different outcome

In layout B, inputs with *identical* building blocks and coefficients sit on the same spot – they compute the same
function the same way and only differ in term order. If the model is right on some of them and wrong on others, that's
the `05` finding (order matters) showing up in the natural data, without any synthetic changes.

In [ ]:
df["block_key"] = [json.dumps(sorted(b.items())) for b in blocks]
groups = df.groupby("block_key").agg(n=("error", "size"), errors=("error", "sum"))
groups = groups[groups.n >= 2]
mixed = groups[(groups.errors > 0) & (groups.errors < groups.n)]
print(f"{len(groups):,} groups of >= 2 test inputs with identical building blocks; "
      f"{len(mixed):,} of them ({len(mixed) / len(groups):.0%}) contain both correct and wrong outputs")
example_key = mixed.sort_values("n", ascending=False).index[0]
df[df.block_key == example_key][["src", "trg", "pred", "verdict"]]

## 8. Interactive explorer

Writes `figures/projection_explorer.html` – open it in a browser (needs internet for plotly.js).
* **Layout:** A (surface form), B (building blocks), or C (wrong outputs only, placed by kind of mistake).
* **Colour:** correct/wrong, region (named), mistake type, robustness, any single change from `05`, how often the
  answer occurred in training, variable pair, …
* **Hover** for details; **lasso- or box-select** a region to list its inputs with error rate and mean robustness.

In [ ]:
def col(values):
    """JSON-safe list (NaN -> null)."""
    return [None if (isinstance(v, float) and np.isnan(v)) else v for v in values]

df["region_s"] = df.r_surface.map(lambda c: reg_s.name.get(c, "no region") if c >= 0 else "no region")
df["region_b"] = df.r_blocks.map(lambda c: reg_b.name.get(c, "no region") if c >= 0 else "no region")
df[["ex", "ey"]] = err[["ex", "ey"]].reindex(df.index)
df["region_e"] = err["e_region"].reindex(df.index).fillna("-")
df["answer_freq_log"] = np.log10(1 + df.trg_train_freq)
df["hover"] = [f"<b>{s}</b><br>answer: {t}<br>model: {p}<br>{m} · answer seen {f}x in training"
               + ("" if np.isnan(r) else f" · robustness {r:.0%}")
               for s, t, p, m, f, r in zip(df.src, df.trg, df.pred, df.mistake, df.trg_train_freq, df.robustness)]

CAT = ["correct", "region", "mistake", "verdict", "answer_kind", "var_pair"] + [f"probe: {p}" for p in PROBE_NAMES]
CONT = {"robustness": "robustness", "answer_freq_log": "log10(1 + answer seen in training)"}
data = {c: col(df[c].tolist()) for c in ["src", "trg", "pred", "verdict", "mistake", "answer_kind", "var_pair",
                                         "region_s", "region_b", "region_e", "hover", "sx", "sy", "bx", "by", "ex", "ey",
                                         "robustness", "answer_freq_log"] + [f"probe: {p}" for p in PROBE_NAMES]}
data["correct"] = df.correct.astype(int).tolist()
options = "".join(f"<option>{c}</option>" for c in CAT) + "".join(f'<option value="{k}">{v} (continuous)</option>'
                                                                   for k, v in CONT.items())

html = """<!doctype html><html><head><meta charset="utf-8"><title>NeuReduce projection explorer</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>body{font-family:system-ui,sans-serif;margin:16px;color:#222} select{margin-right:12px}
#info{margin:8px 0;font-weight:600} table{border-collapse:collapse;font-size:12px;font-family:monospace}
td,th{border-bottom:1px solid #ddd;padding:3px 8px;text-align:left} tr.bad td{color:#b00}</style></head><body>
<h3>NeuReduce test inputs – 2D projections</h3>
Layout <select id="layout"><option value="s">A: surface form</option><option value="b">B: building blocks</option>
<option value="e">C: wrong outputs, by kind of mistake</option></select>
Colour <select id="color">__OPTIONS__</select>
Show <select id="show"><option value="all">all inputs</option><option value="wrong">only wrong</option></select>
<div id="plot" style="height:660px"></div><div id="info">Lasso/box-select points to list them.</div><div id="table"></div>
<script>
const D = __DATA__, N = D.src.length, CONT = __CONT__;
const XY = {s:["sx","sy"], b:["bx","by"], e:["ex","ey"]};
function groups(key, layout){
  if(key==="correct") return D.correct.map(c=>c?"correct":"wrong");
  if(key==="region") return D["region_"+layout];
  return D[key];
}
function draw(){
  const layout=document.getElementById("layout").value, key=document.getElementById("color").value,
        show=document.getElementById("show").value, [xk,yk]=XY[layout];
  const idx=[...Array(N).keys()].filter(i=>D[xk][i]!==null && (show==="all"||!D.correct[i]));
  const mk=(ids,name,marker)=>({type:"scattergl",mode:"markers",name:name+" ("+ids.length+")",
    x:ids.map(i=>D[xk][i]),y:ids.map(i=>D[yk][i]),customdata:ids,text:ids.map(i=>D.hover[i]),
    hovertemplate:"%{text}<extra></extra>",marker:Object.assign({size:4,opacity:0.75},marker)});
  let traces;
  if(key in CONT){
    const has=idx.filter(i=>D[key][i]!==null), no=idx.filter(i=>D[key][i]===null);
    traces=[mk(no,"no value",{color:"#dddddd"}),
            mk(has,CONT[key],{color:has.map(i=>D[key][i]),colorscale:key==="robustness"?"RdYlGn":"Viridis",
                              showscale:true,colorbar:{title:CONT[key]}})];
  } else {
    const g=groups(key,layout), by={};
    idx.forEach(i=>(by[g[i]]=by[g[i]]||[]).push(i));
    traces=Object.keys(by).sort((a,b)=>by[b].length-by[a].length).map(name=>
      mk(by[name],name,key==="correct"?{color:name==="wrong"?"#d62728":"#bbbbbb"}:{}));
  }
  Plotly.newPlot("plot",traces,{dragmode:"lasso",legend:{itemsizing:"constant"},margin:{t:10},
    xaxis:{visible:false},yaxis:{visible:false}},{responsive:true});
  document.getElementById("plot").on("plotly_selected",ev=>{
    if(!ev) return;
    const ids=ev.points.map(p=>p.customdata), wrong=ids.filter(i=>!D.correct[i]).length,
          rob=ids.map(i=>D.robustness[i]).filter(v=>v!==null);
    document.getElementById("info").textContent=ids.length+" selected - error rate "+
      (ids.length?(100*wrong/ids.length).toFixed(1):0)+"% (overall __BASE__%)"+
      (rob.length?" - mean robustness "+(100*rob.reduce((a,b)=>a+b,0)/rob.length).toFixed(0)+"%":"");
    const rows=ids.slice(0,300).map(i=>"<tr class='"+(D.correct[i]?"":"bad")+"'><td>"+D.src[i]+"</td><td>"+D.trg[i]+
      "</td><td>"+D.pred[i]+"</td><td>"+D.mistake[i]+"</td></tr>").join("");
    document.getElementById("table").innerHTML="<table><tr><th>input</th><th>answer</th><th>model</th><th>result</th></tr>"+
      rows+"</table>"+(ids.length>300?"<p>(first 300 shown)</p>":"");
  });
}
["layout","color","show"].forEach(id=>document.getElementById(id).onchange=draw); draw();
</script></body></html>"""
html = (html.replace("__DATA__", json.dumps(data)).replace("__CONT__", json.dumps(CONT))
            .replace("__OPTIONS__", options).replace("__BASE__", f"{100 * BASE:.1f}"))
out = FIG / "projection_explorer.html"
out.write_text(html)
print(f"wrote {out} ({out.stat().st_size / 1e6:.1f} MB) - open it in a browser")

## Writing it up

> *"In a projection of the inputs by their building blocks, errors concentrate in cluster #… (X% vs. 9% overall).
> It is distinguished by … . Inside it, sub-cluster … fails Y% while … fails only Z%, because … .
> Inputs with identical building blocks get different answers in N% of cases – the order of terms matters (`05`)."*